# Feature decisions

Turns the EDA findings in `01_eda.ipynb` into a reproducible set of decisions. The decisions themselves live in `src/credit/features.py`, with the reason for each recorded next to it; this notebook holds the evidence behind them.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("../data/raw")
df = pd.read_csv(DATA_DIR / "application_train.csv")
df.shape

(307511, 122)

## Leakage screen

Univariate ROC-AUC of every numeric feature against TARGET, taken as `max(auc, 1 - auc)` so direction is ignored and only strength counts.

A single feature scoring much above ~0.75 in credit is a warning sign: it usually means the column carries information that only exists *after* the lending decision. Such a model looks excellent in backtest and fails in production, because the column is not there when the decision must be made.

In [2]:
from sklearn.metrics import roc_auc_score

num = df.select_dtypes("number").drop(columns=["SK_ID_CURR", "TARGET"])
auc = {}
for col in num.columns:
    mask = num[col].notna()
    if mask.sum() < 1000 or num.loc[mask, col].nunique() < 2:
        continue
    a = roc_auc_score(df.loc[mask, "TARGET"], num.loc[mask, col])
    auc[col] = max(a, 1 - a)          # arah tidak penting, kekuatan yang penting

auc = pd.Series(auc).sort_values(ascending=False)
auc.head(15)

EXT_SOURCE_3                   0.679400
EXT_SOURCE_1                   0.665681
EXT_SOURCE_2                   0.656120
DAYS_BIRTH                     0.583003
OWN_CAR_AGE                    0.558884
DAYS_LAST_PHONE_CHANGE         0.556864
DAYS_ID_PUBLISH                0.555670
FLOORSMAX_AVG                  0.550385
FLOORSMAX_MEDI                 0.550027
REGION_RATING_CLIENT_W_CITY    0.549182
FLOORSMAX_MODE                 0.549098
REGION_RATING_CLIENT           0.548063
LIVINGAREA_AVG                 0.547116
LIVINGAREA_MEDI                0.546561
TOTALAREA_MODE                 0.545367
dtype: float64

**Result: nothing came close.** Of 104 numeric features, 0 exceed 0.70 and only 3 exceed 0.65 — `EXT_SOURCE_3` (0.679), `EXT_SOURCE_1` (0.666), `EXT_SOURCE_2` (0.656). All three are external risk scores pulled at application time, so being strong is expected, not suspicious. Strong is not the same as leaking; leaking means the information post-dates the decision.

Two side findings:

- `EXT_SOURCE_3` has Pearson r = 0.179 but AUC = 0.679 (Gini 0.359). The same feature looks weak by correlation and respectable by AUC, because AUC is rank-based and unaffected by the 8% class imbalance that caps Pearson. This is why feature selection uses IV/Gini, not r.
- 30 of 104 features (29%) sit at AUC ≤ 0.505 — no univariate signal at all. They are dropped for the additive models in B5 and B9 (logistic, scorecard), where a feature with no standalone separation contributes nothing. This is revisited for B10 (gradient boosting), which finds interactions a univariate screen cannot see.

## Timing audit

The screen above only catches *strong* leakage. Weak leakage passes it, and the
only way to catch that is to ask, per feature family: would the lender know this
at the instant the decision is made?

| Family | Known at application? | Note |
|---|---|---|
| `DAYS_*` | yes | defined as "days before the application" |
| `AMT_*` | yes | the amounts being applied for |
| `EXT_SOURCE_*` | yes | external scores, pulled at application |
| `FLAG_DOCUMENT_*` | yes | documents submitted with the application |
| building block | yes | external property data |
| `SK_ID_CURR` | — | row identifier, not a feature — dropped |

**`REGION_RATING_CLIENT` needs a judgement, not a yes/no.** The data dictionary calls it *"our rating of the region"* — a lender judgement frozen into the data. It is available at application time, so it is not leakage. But if that rating was itself built from past credit outcomes, a model using it learns from prior lending decisions rather than from risk: a feedback loop. Retained for now, flagged in the model card, and a candidate for removal if the scorecard leans on it heavily.